# NIFTY 50 Options: Reproducible Research Walkthrough

**Data provenance:** This notebook intentionally runs on **synthetic options data**. Use the CLI `optionlab analyze` with legitimately sourced NSE spot history and F&O files for real market analysis. **No 18% historical VaR claim is validated here.**

From the root of this repository: `pip install -e .` once, then run the cells below.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from optionlab.cli import main
from optionlab.pricing import black_scholes, crr_binomial, greeks, implied_volatility


## 1. Recreate the 100,000-path study

This regenerates both the market-like inputs and all model outputs with the fixed seed. Option data here are generated, **not historical NSE observations**.


In [ ]:
main(["demo", "--n-paths", "100000", "--seed", "42", "--out", "results/demo"])
summary = json.loads(Path("results/demo/summary.json").read_text())
summary["dataset"], summary["pricing_metrics"]


## 2. Inspect option-pricing residuals and historical-vol regime groups

In [ ]:
scored = pd.read_csv("results/demo/scored_quotes.csv")
regime = pd.read_csv("results/demo/bias_by_regime.csv")
display(regime)
display(scored[["quote_date", "option_type", "spot", "strike", "market_price", "bs_price", "binomial_price", "pricing_error", "delta"]].head(12))


In [ ]:
display(Image(filename="results/demo/figures/regime_bias.png"))
display(Image(filename="results/demo/figures/market_vs_model.png"))


## 3. Greeks and tree convergence for a representative contract

Analytical Vega is **per 100-percentage-point volatility move**. Multiply by 0.01 for a one-percentage-point volatility sensitivity.


In [ ]:
example = dict(spot=23000, strike=23000, maturity=28/365, rate=.065, sigma=.20, kind="CE", dividend=.012)
bs = black_scholes(**example)
print("BSM:", round(bs, 4), "Greeks:", greeks(**example))
steps = [5, 10, 25, 50, 100, 200, 500, 1000]
errors = [abs(crr_binomial(**example, steps=n) - bs) for n in steps]
plt.figure(figsize=(7, 4))
plt.plot(steps, errors, marker="o")
plt.yscale("log")
plt.xlabel("Binomial steps")
plt.ylabel("Absolute deviation from BSM (index pts)")
plt.title("European CRR convergence to analytical Black–Scholes")
plt.show()


## 4. Price risk and delta hedge

`hedge_var` is one-short-call scenario risk, and `historical_hedge_backtest` is a separate quote-pair analysis. The demo uses different 28-day expiries on different quote dates, so it **cannot** create historical quote pairs and returns `unavailable` for that field. When multiple daily records exist for an identical contract in an actual supplied data set, the quote-pair backtest becomes available.


In [ ]:
display(pd.DataFrame([summary["hedge_var"]]).T.rename(columns={0:"value"}))
print("Historical hedge backtest:", summary["historical_hedge_backtest"])
display(Image(filename="results/demo/figures/hedging_var.png"))


## 5. What do these numbers prove?

They show that the implementation is reproducible and the mathematical models are internally consistent. They do **not** demonstrate the original resume's 18% performance number or a real tradeable hedge. See `docs/methodology.md` for the complete risk formula and `docs/interview_walkthrough.md` for concise explanations.
